# Morocco Calibration Workflow**A practical companion to `morocco_calibration_guide.md`.**This notebook walks you through the full calibration process: verify inputs, run the model, diagnose errors, fix parameters, and track progress across iterations.**Pipeline:** `apply_step0_verified.py` → `apply_step1_calibration.py` → `run_calibration0.py` → `compare_to_inventory.py`| I want to... | Go to ||---|---|| Check my inputs before running | Section 1: Preflight Gates || Run the model | Section 2: Pipeline Execution || See what's wrong | Section 3: Diagnostic Dashboard || Explore a specific parameter | Section 4: Parameter Inspector || Fix a calibration error | Section 5: Fix Workflow || Compare across runs | Section 6: Iteration Tracker || Look up reference data | Section 7: Reference Data Browser || Understand sector dependencies | Section 8: DAG Explorer |

In [ ]:
import subprocess, sys, glob, json, os, warningsfrom pathlib import Pathfrom datetime import datetimeimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltwarnings.filterwarnings("ignore")%matplotlib inline# Path detectiontry:    PROJECT_DIR = Path(subprocess.check_output(        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())except Exception:    PROJECT_DIR = Path('.').resolve().parent.parentSSP_DIR = PROJECT_DIR / 'ssp_modeling'INPUT_DIR = SSP_DIR / 'input_data'OUTPUT_DIR = SSP_DIR / 'ssp_run_output'NOTEBOOKS_DIR = SSP_DIR / 'notebooks'SCRIPTS_DIR = SSP_DIR / 'output_postprocessing' / 'scripts'TARGETS_DIR = SSP_DIR / 'output_postprocessing' / 'data' / 'invent'EXTERNAL_DIR = PROJECT_DIR / 'external_data'CONFIG_DIR = SSP_DIR / 'config_files'# Add scripts to path for compare_to_inventory importsys.path.insert(0, str(SCRIPTS_DIR))print(f"Project: {PROJECT_DIR}")print(f"Input:   {INPUT_DIR}")print(f"Output:  {OUTPUT_DIR}")for p in [INPUT_DIR, OUTPUT_DIR, SCRIPTS_DIR, EXTERNAL_DIR]:    assert p.exists(), f"Missing: {p}"print("All paths OK.")

In [ ]:
# Load configimport yamlconfig_path = CONFIG_DIR / 'config.yaml'with open(config_path) as f:    config = yaml.safe_load(f)COUNTRY = config.get('country_name', 'morocco')INPUT_FILE = config.get('ssp_input_file_name', 'df_input_0.csv')ENERGY_FLAG = config.get('energy_model_flag', True)print(f"Country: {COUNTRY}")print(f"Input file: {INPUT_FILE}")print(f"Energy model: {ENERGY_FLAG}")

In [ ]:
# Load input CSV and targetsdf = pd.read_csv(INPUT_DIR / 'df_input_0.csv')print(f"Input CSV: {len(df)} rows x {len(df.columns)} columns")# Find targets filetargets_files = sorted(TARGETS_DIR.glob('emission_targets_*.csv'))TARGETS_FILE = targets_files[-1] if targets_files else Noneif TARGETS_FILE:    targets = pd.read_csv(TARGETS_FILE)    print(f"Targets:  {TARGETS_FILE.name} ({len(targets)} categories)")else:    print("WARNING: No targets file found in", TARGETS_DIR)    targets = None

In [ ]:
# Country parameterizationCOUNTRY_CODE = 'MAR'CALIBRATION_TP = 7   # tp=7 = 2022BASE_YEAR = 2015N_TP = len(df)print(f"Base year: {BASE_YEAR}, Calibration year: {BASE_YEAR + CALIBRATION_TP}, Time periods: {N_TP}")

---# Section 1: Preflight Gates**Why:** Wrong population cascades to every sector. Wrong GDP scale invalidates all energy intensities. These 10 checks take 30 seconds and catch errors that cost hours of model runtime.Each gate compares the input CSV against external data and prints PASS or FAIL.

In [ ]:
# Gate 1: Populationdef check_gate(name, model_val, ref_val, tolerance=0.05):    err = abs(model_val - ref_val) / ref_val    status = "PASS" if err < tolerance else "FAIL"    print(f"  {status:4s} | {name:35s} | Model: {model_val:>15,.0f} | Reference: {ref_val:>15,.0f} | Error: {err:.1%}")    return status == "PASS"results = []print("=" * 100)print("  PREFLIGHT GATES")print("=" * 100)# Gate 1: Populationpop_model = df['population_gnrl_rural'].iloc[0] + df['population_gnrl_urban'].iloc[0]try:    wb_pop = pd.read_csv(EXTERNAL_DIR / 'world_bank' / f'{COUNTRY.lower()}_population_total.csv')    pop_ref = wb_pop[wb_pop.columns[-1]].iloc[-1] if 'Value' not in wb_pop.columns else wb_pop.loc[wb_pop['Year'] == BASE_YEAR, 'Value'].iloc[0]    # Try to find 2015 population    for col in wb_pop.columns:        if str(BASE_YEAR) in str(col):            pop_ref = wb_pop[col].iloc[0]            break    results.append(check_gate("Gate 1: Population", pop_model, pop_ref))except Exception as e:    print(f"  SKIP | Gate 1: Population — {e}")    results.append(None)

In [ ]:
# Gate 2: GDPgdp_model = df['gdp_mmm_usd'].iloc[0]print(f"  INFO | Gate 2: GDP = {gdp_model:.2f} billion USD (verify against World Bank)")# Gate 3: Occupancyocc = df['occrateinit_gnrl_occupancy'].iloc[0] if 'occrateinit_gnrl_occupancy' in df.columns else Noneif occ:    hh_count = pop_model / occ    print(f"  INFO | Gate 3: Occupancy = {occ:.2f} → HH count = {hh_count:,.0f}")# Gate 4: Livestockprint("\n  Gate 4: Livestock Populations")for species in ['cattle_dairy', 'cattle_nondairy', 'sheep', 'goats', 'chickens', 'horses', 'mules', 'pigs']:    col = f'pop_lvst_initial_{species}'    if col in df.columns:        val = df[col].iloc[0]        print(f"    {species:20s}: {val:>15,.0f}")

In [ ]:
# Gate 5: Fertilizerfert_col = 'qtyinit_soil_synthetic_fertilizer_kt'if fert_col in df.columns:    print(f"  INFO | Gate 5: Fertilizer = {df[fert_col].iloc[0]:.1f} kt N")# Gate 7b: Fuel Exports (should be zero for net importers)print("\n  Gate 7b: Fuel Exports (should be 0 for net importers)")for col in sorted(c for c in df.columns if c.startswith('exports_enfu_pj_fuel_')):    val = df[col].iloc[0]    status = "PASS" if abs(val) < 0.01 else "WARN"    fuel = col.replace('exports_enfu_pj_fuel_', '')    if abs(val) > 0.01:        print(f"    {status} | {fuel:25s}: {val:.2f} PJ")# Gate 7c: Wasteprint("\n  Gate 7c: Waste Baseline")for col in ['qty_waso_initial_municipal_waste_tonne_per_capita', 'frac_waso_landfill_gas_recovered']:    if col in df.columns:        name = col.replace('qty_waso_initial_', '').replace('frac_waso_', '')        print(f"    {name:40s}: {df[col].iloc[0]:.4f}")# Gate 10: inf/NaNn_inf = sum(np.isinf(df[c]).sum() for c in df.select_dtypes(include=[np.number]).columns)n_nan = df.isna().sum().sum()status = "PASS" if n_inf == 0 and n_nan == 0 else "FAIL"print(f"\n  {status} | Gate 10: inf={n_inf}, NaN={n_nan}")

---# Section 2: Pipeline Execution**The three-script pipeline:**1. `apply_step0_verified.py` — Build df_input_0.csv from raw template + external data2. `apply_step1_calibration.py` — Apply sector-specific calibration fixes3. `run_calibration0.py` — Run SISEPUEDE model + diagnosticsRun step0+step1 first, then choose --no-energy (fast, ~30s) or full mode (~5 min).

In [ ]:
# Run step0 + step1PYTHON = sys.executabledef run_script(name, script_path):    print(f"Running {name}...")    result = subprocess.run([PYTHON, str(script_path)], capture_output=True, text=True, cwd=str(PROJECT_DIR))    if result.returncode != 0:        print(f"  ERROR: {result.stderr[-500:]}")    else:        # Show last 5 lines of output        lines = result.stdout.strip().split('\n')        for line in lines[-5:]:            print(f"  {line}")    return result.returncode == 0run_script("apply_step0_verified", NOTEBOOKS_DIR / "apply_step0_verified.py")run_script("apply_step1_calibration", NOTEBOOKS_DIR / "apply_step1_calibration.py")

In [ ]:
# Run calibration (choose one)# Fast mode: --no-energy (~30 sec, tests AFOLU/CircularEconomy/IPPU only)# Full mode: includes NemoMod energy optimization (~3-5 min)MODE = "--baseline-only"  # Change to "--baseline-only --no-energy" for fast moderesult = subprocess.run(    [PYTHON, str(NOTEBOOKS_DIR / "run_calibration0.py"), MODE, "--input-file", "df_input_0.csv"],    capture_output=True, text=True, cwd=str(PROJECT_DIR), timeout=600)lines = result.stdout.strip().split('\n')for line in lines[-20:]:    print(line)

In [ ]:
# Find latest run outputruns = sorted(OUTPUT_DIR.glob('calibration_*'))if runs:    LATEST_RUN = runs[-1]    WIDE_PATH = LATEST_RUN / 'WIDE_INPUTS_OUTPUTS.csv'    print(f"Latest run: {LATEST_RUN.name}")    print(f"WIDE file:  {'EXISTS' if WIDE_PATH.exists() else 'MISSING'}")    if WIDE_PATH.exists():        wide = pd.read_csv(WIDE_PATH)        print(f"  {len(wide)} rows x {len(wide.columns)} columns")else:    print("No calibration runs found.")    LATEST_RUN = None

---# Section 3: Diagnostic DashboardImport the `compare_to_inventory` API to run diagnostics directly in the notebook. Returns DataFrames you can visualize and filter.**How to read results:**- `error_pct` = |model - inventory| / |inventory| × 100- `direction` = "over" (model too high) or "under" (model too low)- `top_component` = the single largest model output variable in that category

In [ ]:
from compare_to_inventory import compare, DiagnosticConfig, DAG_AFFECTSconfig = DiagnosticConfig(tp=CALIBRATION_TP, threshold=0.15)diff, flagged, diag = compare(str(TARGETS_FILE), str(WIDE_PATH), config, verbose=False)sig = diff[diff['inventory'].abs() > 0.01]total_err = sig['diff'].abs().sum()n_15 = (sig['error_pct'] <= 15).sum()n_25 = (sig['error_pct'] <= 25).sum()status = "GOOD" if total_err < 10 and n_15 > len(sig) * 0.5 else "NEEDS WORK"print(f"CALIBRATION: {total_err:.2f} MtCO2e | {n_15}/{len(sig)} within 15% | {status}")

In [ ]:
# Sector totals bar chartsector_data = []for sec in sorted(diff['sector'].unique()):    sub = diff[diff['sector'] == sec]    sector_data.append({'Sector': sec, 'Inventory': sub['inventory'].sum(), 'Model': sub['model'].sum()})sdf = pd.DataFrame(sector_data)sdf['Diff'] = sdf['Model'] - sdf['Inventory']fig, ax = plt.subplots(figsize=(12, 5))x = range(len(sdf))w = 0.35ax.bar([i - w/2 for i in x], sdf['Inventory'], w, label='Inventory (NIR)', color='steelblue')ax.bar([i + w/2 for i in x], sdf['Model'], w, label='Model', color='coral')ax.set_xticks(x)ax.set_xticklabels(sdf['Sector'], rotation=30, ha='right', fontsize=9)ax.set_ylabel('MtCO2e')ax.set_title('Sector Totals: Inventory vs Model')ax.legend()ax.axhline(0, color='gray', linewidth=0.5)plt.tight_layout()plt.show()

In [ ]:
# Top flagged categories tabletop = flagged.head(10)[['ID', 'category', 'gas', 'inventory', 'model', 'diff', 'error_pct', 'direction', 'top_component']].copy()top['error_pct'] = top['error_pct'].round(1)top['diff'] = top['diff'].round(3)print("Top 10 Flagged Categories (sorted by absolute error):\n")print(top.to_string(index=False))

In [ ]:
# Structural diagnosticsif len(diag) > 0:    print("Diagnostics:\n")    for sev in ['HIGH', 'MEDIUM']:        sev_df = diag[diag['severity'] == sev]        if len(sev_df) > 0:            print(f"  {sev} ({len(sev_df)}):")            for _, w in sev_df.iterrows():                print(f"    {w['issue']:30s} {w['ID']:25s} {str(w['detail'])[:60]}")            print()else:    print("No diagnostics (all clean).")

In [ ]:
# Category inspector — inspect any category in detaildef inspect_category(category_id):    row = diff[diff['ID'] == category_id]    if len(row) == 0:        print(f"Category '{category_id}' not found. Available: {', '.join(diff['ID'].tolist()[:10])}...")        return    row = row.iloc[0]    print(f"Category: {row['ID']}  ({row.get('category', '')})")    print(f"  Inventory: {row['inventory']:.3f} MtCO2e")    print(f"  Model:     {row['model']:.3f} MtCO2e")    print(f"  Diff:      {row['diff']:+.3f} ({row['error_pct']:.1f}% {row['direction']})")    print(f"  Top component: {row['top_component']}")    # Show all components    from compare_to_inventory import get_components, load_model_row    model_row = load_model_row(WIDE_PATH, CALIBRATION_TP)    comps = get_components(row.get('vars', ''), model_row, top_n=10)    if len(comps) > 0:        print(f"  Components:")        for _, c in comps.iterrows():            from compare_to_inventory import short_name            pct = c['val'] / row['model'] * 100 if abs(row['model']) > 1e-6 else 0            print(f"    {short_name(c['var']):50s} {c['val']:8.4f}  ({pct:5.1f}%)")# Example: inspect_category("3.A.1:CH4")inspect_category("3.A.1:CH4")

---# Section 4: Parameter InspectorThe input CSV has ~2,420 columns. These tools help you find, inspect, and validate specific parameters.

In [ ]:
def search_columns(keyword, tp=0):    """Search for columns containing a keyword and show their values."""    matches = [c for c in df.columns if keyword.lower() in c.lower()]    if not matches:        print(f"No columns matching '{keyword}'")        return    print(f"{len(matches)} columns matching '{keyword}':")    for c in sorted(matches)[:20]:        print(f"  {c:70s} tp0={df[c].iloc[0]:.6g}  tp7={df[c].iloc[CALIBRATION_TP]:.6g}")    if len(matches) > 20:        print(f"  ... and {len(matches)-20} more")search_columns("cement")

In [ ]:
def validate_fraction_group(prefix):    """Check that all columns with this prefix sum to 1.0 at every time period."""    cols = [c for c in df.columns if c.startswith(prefix)]    if not cols:        print(f"No columns matching prefix '{prefix}'")        return    sums = df[cols].sum(axis=1)    ok = sums.between(0.999, 1.001).all()    print(f"Fraction group: {prefix}*  ({len(cols)} members)")    print(f"  Sum range: {sums.min():.6f} – {sums.max():.6f}")    print(f"  Status: {'PASS' if ok else 'FAIL — sums deviate from 1.0'}")    if not ok:        bad_tps = sums[~sums.between(0.999, 1.001)].index.tolist()        print(f"  Problem at time periods: {bad_tps[:5]}")    return colsvalidate_fraction_group("frac_inen_energy_cement_")

In [ ]:
def plot_parameter(column_name):    """Plot a parameter over all time periods."""    if column_name not in df.columns:        print(f"Column '{column_name}' not found")        return    fig, ax = plt.subplots(figsize=(10, 4))    ax.plot(range(N_TP), df[column_name], 'b-o', markersize=3)    ax.axvline(CALIBRATION_TP, color='red', linestyle='--', alpha=0.5, label=f'tp={CALIBRATION_TP}')    ax.set_xlabel('Time Period')    ax.set_ylabel(column_name.split('_')[-1])    ax.set_title(column_name)    ax.legend()    plt.tight_layout()    plt.show()plot_parameter("gdp_mmm_usd")

---# Section 5: Fix Workflow**The golden rule: Scale, Don't Replace.**`factor = target_value / current_value_at_tp7`Multiply the entire column by this factor. This preserves the trajectory shape while correcting the level.

In [ ]:
def calculate_scaling(column, target_value, ref_tp=CALIBRATION_TP):    """Preview a scaling factor without applying it."""    current = df[column].iloc[ref_tp]    if abs(current) < 1e-10:        print(f"  WARNING: {column} is near zero at tp={ref_tp}. Cannot scale.")        return None    factor = target_value / current    print(f"  Column:  {column}")    print(f"  Current: {current:.6g} at tp={ref_tp}")    print(f"  Target:  {target_value:.6g}")    print(f"  Factor:  {factor:.4f}")    if abs(factor) > 10 or abs(factor) < 0.1:        print(f"  WARNING: factor {factor:.2f} is extreme (>10x or <0.1x). Double-check.")    return factor# Example: calculate_scaling("pop_lvst_initial_sheep", 20_000_000)

In [ ]:
def apply_scaling(column, target_value, ref_tp=CALIBRATION_TP):    """Apply scaling factor to preserve trajectory shape."""    factor = calculate_scaling(column, target_value, ref_tp)    if factor is not None:        df[column] *= factor        print(f"  Applied. New value at tp={ref_tp}: {df[column].iloc[ref_tp]:.6g}")    return factordef rebalance_fractions(prefix, target_col, target_value):    """Set one fraction and proportionally rescale others to sum to 1.0."""    cols = [c for c in df.columns if c.startswith(prefix)]    others = [c for c in cols if c != target_col]    old_others_sum = df[others].sum(axis=1)    df[target_col] = target_value    mask = old_others_sum > 1e-10    df.loc[mask, others] = df.loc[mask, others].mul((1 - target_value) / old_others_sum[mask], axis=0)    df.loc[~mask, others] = (1 - target_value) / len(others)    new_sum = df[cols].sum(axis=1)    print(f"  Rebalanced {prefix}*: sum range {new_sum.min():.6f} – {new_sum.max():.6f}")

In [ ]:
def save_input_csv():    """Save df_input_0.csv with backup."""    csv_path = INPUT_DIR / 'df_input_0.csv'    backup = INPUT_DIR / f'df_input_0.csv.bak_{datetime.now().strftime("%Y%m%d_%H%M%S")}'    if csv_path.exists():        import shutil        shutil.copy(csv_path, backup)        print(f"  Backup: {backup.name}")    n_inf = sum(np.isinf(df[c]).sum() for c in df.select_dtypes(include=[np.number]).columns)    n_nan = df.isna().sum().sum()    print(f"  Validation: inf={n_inf}, NaN={n_nan}")    if n_nan > 0:        df.fillna(0, inplace=True)        print(f"  Filled {n_nan} NaN values with 0")    df.to_csv(csv_path, index=False)    print(f"  Saved to {csv_path.name}")# Uncomment to save: save_input_csv()

---# Section 6: Iteration TrackerTrack calibration progress across multiple runs. Detect regressions (a sector getting worse after a fix).**Stopping rules:**- No fix improves total error by > 0.5 MtCO2e- Oscillation detected (sector improves then worsens)- All remaining gaps are structural (documented)

In [ ]:
# List all calibration runsruns = sorted(OUTPUT_DIR.glob('calibration_*'))print(f"Found {len(runs)} calibration runs:")progress = []for run_dir in runs[-10:]:    diag_file = run_dir / 'diagnostics' / 'diff_report.csv'    if diag_file.exists():        d = pd.read_csv(diag_file)        sig = d[d['inventory'].abs() > 0.01]        total = sig['diff'].abs().sum()        n15 = (sig['error_pct'] <= 15).sum()        progress.append({'run': run_dir.name, 'total_error': total, 'within_15pct': n15, 'categories': len(sig)})        print(f"  {run_dir.name}: {total:.2f} MtCO2e | {n15}/{len(sig)} within 15%")    else:        print(f"  {run_dir.name}: (no diagnostics)")if progress:    prog_df = pd.DataFrame(progress)

In [ ]:
# Progress chartif progress and len(progress) > 1:    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))    ax1.plot(range(len(progress)), [p['total_error'] for p in progress], 'bo-')    ax1.axhline(15, color='green', linestyle='--', alpha=0.5, label='Target (15 MtCO2e)')    ax1.set_xlabel('Iteration')    ax1.set_ylabel('Total Error (MtCO2e)')    ax1.set_title('Calibration Progress')    ax1.legend()    ax2.bar(range(len(progress)), [p['within_15pct'] for p in progress], color='steelblue')    ax2.set_xlabel('Iteration')    ax2.set_ylabel('Categories within 15%')    ax2.set_title('Categories Meeting Threshold')    plt.tight_layout()    plt.show()else:    print("Need 2+ runs for progress chart.")

---# Section 7: Reference Data Browser**Reference hierarchy:**1. NIR / National Inventory (authoritative for targets)2. SNBC / NDC (official BAU projections)3. IEA (energy data)4. EDGAR v8.0 (cross-check only — known errors in waste CH4, soil N2O, HFC)5. FAO / World Bank (socioeconomic)6. IPCC tables (emission factors, MCFs)

In [ ]:
# FAO Livestockfao_files = sorted(EXTERNAL_DIR.glob('fao/*.csv'))print(f"FAO files: {len(fao_files)}")for f in fao_files:    print(f"  {f.name}")if any('livestock' in f.name.lower() and 'pattern' in f.name.lower() for f in fao_files):    livestock_file = [f for f in fao_files if 'livestock' in f.name.lower() and 'pattern' in f.name.lower()][0]    fao_livestock = pd.read_csv(livestock_file)    print(f"\nFAO Livestock Patterns: {len(fao_livestock)} rows")    print(fao_livestock.head())

In [ ]:
# IPCC tablesipcc_dir = PROJECT_DIR / 'ipcc_tables'if ipcc_dir.exists():    ipcc_files = sorted(ipcc_dir.glob('*.csv'))    print(f"IPCC tables: {len(ipcc_files)}")    for f in ipcc_files:        print(f"  {f.name}")else:    print("No ipcc_tables/ directory found")

In [ ]:
# Targets summaryif targets is not None:    print(f"Inventory targets ({TARGETS_FILE.name}):\n")    for sec in sorted(targets['sector'].unique()):        sub = targets[targets['sector'] == sec]        total = sub['inventory'].sum()        print(f"  {sec:35s}: {total:8.3f} MtCO2e ({len(sub)} categories)")    print(f"  {'TOTAL':35s}: {targets['inventory'].sum():8.3f} MtCO2e")

---# Section 8: DAG ExplorerSISEPUEDE runs sectors in a fixed order. Changing parameters in one sector can cascade to downstream sectors.```Socioeconomic (population, GDP, HH count) → feeds ALL sectorsStep 1: AFOLU (LNDU → FRST → AGRC → LVST → LSMM → SOIL)  → CircularEconomy (food loss → MSW), EnergyConsumption (crop yield), EnergyProduction (biogas)Step 2: CircularEconomy (WALI → TRWW → WASO) → IPPU, EnergyProductionStep 3: IPPU (cement, HFC, metals) → EnergyConsumption (production volumes)Step 4: EnergyConsumption (CCSQ → INEN → SCOE → TRNS) → EnergyProduction (fuel demand)Step 5: EnergyProduction (NemoMod LP: ENTC + ENFU) → FugitiveEmissionsStep 6: FugitiveEmissions (FGTV) — terminal```**Before fixing any sector, trace upstream (is the error inherited?) and downstream (what else will move?).**

In [ ]:
from compare_to_inventory import DAG_AFFECTSdef trace_upstream(subsector):    """What feeds this subsector?"""    upstream = [k for k, v in DAG_AFFECTS.items() if subsector in v]    print(f"Upstream of {subsector}: {upstream if upstream else '(none — root sector)'}")    return upstreamdef trace_downstream(subsector):    """What does this subsector affect?"""    downstream = DAG_AFFECTS.get(subsector, [])    print(f"Downstream of {subsector}: {downstream if downstream else '(terminal node)'}")    return downstreamtrace_upstream("entc")trace_downstream("entc")print()trace_upstream("lvst")trace_downstream("lvst")

In [ ]:
def predict_cascade(subsector):    """Show the full cascade from a subsector change."""    visited = set()    queue = [subsector]    path = []    while queue:        current = queue.pop(0)        if current in visited:            continue        visited.add(current)        affected = DAG_AFFECTS.get(current, [])        for a in affected:            path.append(f"  {current} → {a}")            queue.append(a)    if path:        print(f"Cascade from changing {subsector}:")        for p in path:            print(p)    else:        print(f"{subsector} is a terminal node (no downstream effects)")predict_cascade("lvst")

---*End of calibration workflow notebook. See `morocco_calibration_guide.md` for the full theoretical background.*